# Seasonal ARIMA (SARIMA)

Many time series have both **trend** and **seasonality**.  Standard ARIMA
handles trends through differencing, but it does not account for seasonal
patterns.  **Seasonal ARIMA** (SARIMA) extends ARIMA with additional
seasonal terms.

A SARIMA model is written as:

$$
\text{ARIMA}(p,d,q)(P,D,Q)_m
$$

where:

| | Non-seasonal | Seasonal |
|---|---|---|
| AR order | $p$ | $P$ |
| Differencing | $d$ | $D$ |
| MA order | $q$ | $Q$ |
| Period | — | $m$ (e.g., 12 for monthly, 4 for quarterly) |

In backshift notation:

$$
\Phi_p(B)\,\tilde{\Phi}_P(B^m)\,(1-B)^d\,(1-B^m)^D\,y_t = c + \Theta_q(B)\,\tilde{\Theta}_Q(B^m)\,\varepsilon_t
$$

This notebook covers:
1. Seasonal differencing
2. The SARIMAX API in statsmodels
3. Complete walkthrough on airline passengers
4. Automatic seasonal order selection with `auto_arima`
5. Application to CO2 data

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Load Airline Passengers Data

In [ ]:
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

print(f"Shape: {airline.shape}")
print(f"Date range: {airline.index[0].date()} to {airline.index[-1].date()}")
airline.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(airline["Passengers"])
ax.set_title("International Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

print("Clear upward trend + increasing seasonal amplitude.")
print("This series has both trend and multiplicative seasonality.")
print("We will need both regular and seasonal differencing.")

---
## 2. STL Decomposition

Before modelling, let us decompose the series to understand its components.

In [ ]:
stl = STL(airline["Passengers"], period=12, robust=True)
stl_result = stl.fit()

fig = stl_result.plot()
fig.set_size_inches(14, 10)
plt.suptitle("STL Decomposition — Airline Passengers", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Trend: clear upward movement.")
print("Seasonal: regular 12-month pattern with increasing amplitude.")
print("Remainder: relatively small — most variation is explained by trend + season.")

---
## 3. Stationarity Tests

In [ ]:
def stationarity_tests(series, name=""):
    """Run ADF and KPSS tests and print results."""
    adf_stat, adf_p, *_ = adfuller(series.dropna(), autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression="c", nlags="auto")

    print(f"=== {name} ===")
    print(f"  ADF:  stat={adf_stat:.4f}, p={adf_p:.6f}  "
          f"{'Stationary' if adf_p < 0.05 else 'Non-stationary'}")
    print(f"  KPSS: stat={kpss_stat:.4f}, p={kpss_p:.4f}  "
          f"{'Non-stationary' if kpss_p < 0.05 else 'Stationary'}")
    print()


stationarity_tests(airline["Passengers"], "Original")

In [ ]:
print("Clearly non-stationary.  Let us try differencing.")

---
## 4. Seasonal Differencing

**Seasonal differencing** at period $m$:

$$
(1 - B^m) y_t = y_t - y_{t-m}
$$

For monthly data with $m = 12$: $y_t - y_{t-12}$ removes the seasonal pattern
by comparing each month to the same month last year.

**Combined differencing** (regular + seasonal):

$$
(1 - B)(1 - B^{12}) y_t
$$

This removes both the trend (via $1 - B$) and the seasonal pattern (via $1 - B^{12}$).

In [ ]:
# Apply different types of differencing
y = airline["Passengers"]

# Regular first difference
y_d1 = y.diff().dropna()

# Seasonal difference (lag 12)
y_sd = y.diff(12).dropna()

# Both: regular + seasonal
y_d1_sd = y.diff(12).diff().dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(y)
axes[0, 0].set_title("Original")

axes[0, 1].plot(y_d1)
axes[0, 1].set_title("First Difference $(1-B)y_t$")
axes[0, 1].axhline(0, color="grey", linestyle="--", alpha=0.5)

axes[1, 0].plot(y_sd)
axes[1, 0].set_title("Seasonal Difference $(1-B^{12})y_t$")
axes[1, 0].axhline(0, color="grey", linestyle="--", alpha=0.5)

axes[1, 1].plot(y_d1_sd)
axes[1, 1].set_title("Both $(1-B)(1-B^{12})y_t$")
axes[1, 1].axhline(0, color="grey", linestyle="--", alpha=0.5)

for ax in axes.flat:
    ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

In [ ]:
# Test stationarity of each
stationarity_tests(y_d1, "First Difference")
stationarity_tests(y_sd, "Seasonal Difference")
stationarity_tests(y_d1_sd, "Regular + Seasonal Difference")

In [ ]:
# Use pmdarima to determine d and D
from pmdarima.arima import ndiffs, nsdiffs

d = ndiffs(y, test="adf")
D = nsdiffs(y, m=12, test="ocsb")

print(f"Recommended d (regular differencing): {d}")
print(f"Recommended D (seasonal differencing, m=12): {D}")

---
## 5. ACF/PACF of the Differenced Series

After differencing, we examine the ACF and PACF to identify tentative
AR and MA orders (both regular and seasonal).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_d1_sd, lags=36, ax=axes[0], title="ACF — $(1-B)(1-B^{12})y_t$")
plot_pacf(y_d1_sd, lags=36, ax=axes[1], title="PACF — $(1-B)(1-B^{12})y_t$", method="ywm")
plt.tight_layout()
plt.show()

print("Look for patterns at both non-seasonal lags (1, 2, 3, ...) and")
print("seasonal lags (12, 24, 36, ...).")
print()
print("Non-seasonal: ACF/PACF behaviour at lags 1-3 suggests p, q.")
print("Seasonal: spikes at lag 12, 24 suggest P, Q.")

---
## 6. Train / Test Split

In [ ]:
TRAIN_SIZE = 120  # 10 years
train = airline.iloc[:TRAIN_SIZE]
test = airline.iloc[TRAIN_SIZE:]
HORIZON = len(test)  # 24 months

print(f"Train: {len(train)} months ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} months ({test.index[0].date()} to {test.index[-1].date()})")

fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Train")
ax.plot(test["Passengers"], label="Test", color="tab:orange")
ax.axvline(test.index[0], color="grey", linestyle="--", alpha=0.7)
ax.set_title("Train / Test Split")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Fitting SARIMA with SARIMAX

The statsmodels class for seasonal ARIMA is `SARIMAX` (which also supports
exogenous regressors, but we will not use that feature here).

```python
from statsmodels.tsa.statespace.sarimax import SARIMAX
model = SARIMAX(train, order=(p,d,q), seasonal_order=(P,D,Q,m))
result = model.fit(disp=False)
```

In [ ]:
# Try a few candidate SARIMA models
candidates = [
    ((1, 1, 0), (1, 1, 0, 12)),
    ((0, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 1), (1, 1, 1, 12)),
    ((1, 1, 1), (0, 1, 1, 12)),
    ((2, 1, 1), (0, 1, 1, 12)),
    ((0, 1, 1), (1, 1, 0, 12)),
]

results_table = []
for order, seasonal_order in candidates:
    try:
        mod = SARIMAX(
            train["Passengers"],
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False)
        results_table.append({
            "Order": f"{order}x{seasonal_order}",
            "AIC": round(mod.aic, 2),
            "BIC": round(mod.bic, 2),
        })
    except Exception as e:
        print(f"  {order}x{seasonal_order} failed: {e}")

comp_df = pd.DataFrame(results_table).sort_values("AIC").reset_index(drop=True)
print(comp_df.to_string(index=False))
print(f"\nBest by AIC: {comp_df.iloc[0]['Order']}")

In [ ]:
# Fit the best manual model
sarima_model = SARIMAX(
    train["Passengers"],
    order=(0, 1, 1),
    seasonal_order=(0, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_result = sarima_model.fit(disp=False)
print(sarima_result.summary())

---
## 8. Automatic Order Selection with `auto_arima`

In [ ]:
from pmdarima import auto_arima

auto_model = auto_arima(
    train["Passengers"],
    seasonal=True,
    m=12,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
    information_criterion="aic",
)

print(f"\nBest order: {auto_model.order}")
print(f"Best seasonal order: {auto_model.seasonal_order}")
print(auto_model.summary())

---
## 9. Residual Diagnostics

In [ ]:
# Use the sarima_result for diagnostics
fig = sarima_result.plot_diagnostics(figsize=(14, 10))
plt.suptitle("SARIMA Residual Diagnostics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box test on residuals
residuals = sarima_result.resid
lb_test = acorr_ljungbox(residuals, lags=[12, 24, 36], return_df=True)
print("Ljung-Box test on SARIMA residuals:")
print(lb_test)
print()
print("p-values > 0.05 → residuals are white noise — model is adequate.")

---
## 10. Forecasting with Prediction Intervals

In [ ]:
# Forecast from SARIMAX model
fc_obj = sarima_result.get_forecast(steps=HORIZON)
fc_mean = fc_obj.predicted_mean
fc_ci = fc_obj.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_mean, label="SARIMA forecast", color="tab:red", linestyle="--")
ax.fill_between(
    fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
    color="tab:red", alpha=0.15, label="95% CI",
)
ax.set_title("SARIMA Forecast — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation
actual = test["Passengers"].values
pred = fc_mean.values

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
mape = np.mean(np.abs((actual - pred) / actual)) * 100

print("SARIMA(0,1,1)(0,1,1)[12] Test-Set Accuracy:")
print(f"  MAE:  {mae:.2f}")
print(f"  RMSE: {rmse:.2f}")
print(f"  MAPE: {mape:.2f}%")

In [ ]:
# Also get auto_arima forecast for comparison
auto_fc, auto_ci = auto_model.predict(
    n_periods=HORIZON, return_conf_int=True, alpha=0.05
)

auto_mae = mean_absolute_error(actual, auto_fc)
auto_rmse = np.sqrt(mean_squared_error(actual, auto_fc))
auto_mape = np.mean(np.abs((actual - auto_fc) / actual)) * 100

print(f"auto_arima {auto_model.order}x{auto_model.seasonal_order} Accuracy:")
print(f"  MAE:  {auto_mae:.2f}")
print(f"  RMSE: {auto_rmse:.2f}")
print(f"  MAPE: {auto_mape:.2f}%")

---
## 11. Comparison: SARIMA vs Seasonal Naive

In [ ]:
# Seasonal naive benchmark
last_season = train["Passengers"].values[-12:]
snaive_fc = np.tile(last_season, int(np.ceil(HORIZON / 12)))[:HORIZON]
snaive_mae = mean_absolute_error(actual, snaive_fc)
snaive_rmse = np.sqrt(mean_squared_error(actual, snaive_fc))
snaive_mape = np.mean(np.abs((actual - snaive_fc) / actual)) * 100

comparison = pd.DataFrame({
    "Model": [
        "SARIMA(0,1,1)(0,1,1)[12]",
        f"auto_arima {auto_model.order}x{auto_model.seasonal_order}",
        "Seasonal Naive",
    ],
    "MAE": [mae, auto_mae, snaive_mae],
    "RMSE": [rmse, auto_rmse, snaive_rmse],
    "MAPE": [mape, auto_mape, snaive_mape],
})
comparison = comparison.round(2).sort_values("MAE").reset_index(drop=True)
print(comparison.to_string(index=False))
print(f"\nSARIMA substantially outperforms the seasonal naive benchmark.")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test["Passengers"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(test.index, pred, label="SARIMA", color="tab:red", linestyle="--")
ax.plot(test.index, auto_fc, label="auto_arima", color="tab:green", linestyle="--")
ax.plot(test.index, snaive_fc, label="Seasonal Naive", color="tab:purple", linestyle=":")
ax.set_title("Forecast Comparison — Airline Passengers")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 12. Application to CO2 Data

The Mauna Loa CO2 dataset has a strong upward trend and a clear annual
seasonal cycle — a classic candidate for SARIMA.

In [ ]:
co2 = pd.read_csv(DATA_DIR / "co2_mm_mlo.csv")

# Build a proper datetime index from year and month columns
co2["date"] = pd.to_datetime(co2[["year", "month"]].assign(day=1))
co2 = co2.set_index("date").sort_index()

# Use the 'interpolated' column (no missing values)
co2 = co2[["interpolated"]].rename(columns={"interpolated": "CO2"})
co2.index.freq = "MS"

# Drop any rows where CO2 is 0 or negative (missing data encoded as -99.99)
co2 = co2[co2["CO2"] > 0]

print(f"Shape: {co2.shape}")
print(f"Date range: {co2.index[0].date()} to {co2.index[-1].date()}")
co2.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(co2["CO2"], linewidth=0.8)
ax.set_title("Mauna Loa CO2 Concentration")
ax.set_ylabel("CO2 (ppm)")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# Train/test split: hold out last 24 months
co2_train = co2.iloc[:-24]
co2_test = co2.iloc[-24:]

print(f"Train: {len(co2_train)} months")
print(f"Test:  {len(co2_test)} months")

In [ ]:
# auto_arima on CO2
co2_auto = auto_arima(
    co2_train["CO2"],
    seasonal=True,
    m=12,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
)

print(f"\nBest order: {co2_auto.order}")
print(f"Best seasonal order: {co2_auto.seasonal_order}")

In [ ]:
# Forecast CO2
co2_fc, co2_ci = co2_auto.predict(
    n_periods=24, return_conf_int=True, alpha=0.05
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(co2_train["CO2"].iloc[-120:], label="Train (last 10 yr)", color="black", alpha=0.5)
ax.plot(co2_test["CO2"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(co2_test.index, co2_fc, label=f"SARIMA {co2_auto.order}x{co2_auto.seasonal_order}",
        color="tab:red", linestyle="--")
ax.fill_between(co2_test.index, co2_ci[:, 0], co2_ci[:, 1],
                color="tab:red", alpha=0.15, label="95% CI")
ax.set_title("SARIMA Forecast — CO2 Concentration")
ax.set_ylabel("CO2 (ppm)")
ax.legend()
plt.tight_layout()
plt.show()

co2_mae = mean_absolute_error(co2_test["CO2"].values, co2_fc)
co2_rmse = np.sqrt(mean_squared_error(co2_test["CO2"].values, co2_fc))
print(f"CO2 SARIMA — MAE: {co2_mae:.2f} ppm, RMSE: {co2_rmse:.2f} ppm")

---
## 13. The "Airline Model": ARIMA(0,1,1)(0,1,1)[12]

The model ARIMA(0,1,1)(0,1,1)$_{12}$ is so commonly fitted to monthly
data that it is known as the **airline model** (after Box and Jenkins'
original analysis of the airline passengers data).

In full:

$$
(1-B)(1-B^{12})y_t = (1 + \theta_1 B)(1 + \Theta_1 B^{12})\varepsilon_t
$$

This model:
- First-differences to remove trend ($d=1$)
- Seasonally-differences to remove seasonality ($D=1$)
- Has one non-seasonal MA term ($q=1$) and one seasonal MA term ($Q=1$)
- Has no AR terms

It is parsimonious (only 2 parameters + variance) and often fits monthly
data surprisingly well.

In [ ]:
print("The 'airline model' ARIMA(0,1,1)(0,1,1)[12] remains a strong baseline")
print("for monthly seasonal data.  When in doubt, try this model first.")

---
## 14. Log Transformation for Multiplicative Seasonality

When the seasonal amplitude grows with the level (multiplicative seasonality),
a **log transformation** can stabilise the variance, making the additive
SARIMA model more appropriate.

In [ ]:
# Fit SARIMA on log-transformed airline data
log_train = np.log(train["Passengers"])

log_model = SARIMAX(
    log_train,
    order=(0, 1, 1),
    seasonal_order=(0, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

# Forecast in log-space, then back-transform
log_fc = log_model.get_forecast(steps=HORIZON)
log_fc_mean = np.exp(log_fc.predicted_mean)
log_fc_ci = np.exp(log_fc.conf_int(alpha=0.05))

log_mae = mean_absolute_error(actual, log_fc_mean.values)
log_rmse = np.sqrt(mean_squared_error(actual, log_fc_mean.values))
log_mape = np.mean(np.abs((actual - log_fc_mean.values) / actual)) * 100

print("SARIMA on log(Passengers):")
print(f"  MAE:  {log_mae:.2f}")
print(f"  RMSE: {log_rmse:.2f}")
print(f"  MAPE: {log_mape:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test["Passengers"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(test.index, pred, label="SARIMA (raw)", color="tab:red", linestyle="--")
ax.plot(test.index, log_fc_mean.values, label="SARIMA (log)", color="tab:green", linestyle="--")
ax.fill_between(
    log_fc_ci.index, log_fc_ci.iloc[:, 0], log_fc_ci.iloc[:, 1],
    color="tab:green", alpha=0.1, label="95% CI (log)",
)
ax.set_title("SARIMA: Raw vs Log-Transformed")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("The log transformation often improves forecasts when seasonal")
print("amplitude is proportional to the level.")

---
## Summary

- **SARIMA$(p,d,q)(P,D,Q)_m$** extends ARIMA with seasonal AR, differencing,
  and MA terms at the seasonal period $m$.
- **Seasonal differencing** $(1 - B^m)y_t = y_t - y_{t-m}$ removes seasonal
  patterns by comparing each period to the same period in the previous cycle.
- **Combined differencing** $(1-B)(1-B^m)$ handles both trend and seasonality.
- The **airline model** ARIMA$(0,1,1)(0,1,1)_{12}$ is a strong, parsimonious
  baseline for monthly seasonal data.
- Use `statsmodels.tsa.statespace.sarimax.SARIMAX` for fitting.
- `pmdarima.auto_arima(seasonal=True, m=12)` automates order selection.
- A **log transformation** can handle multiplicative seasonality by
  stabilising the variance before fitting the additive SARIMA model.
- Always check residuals (diagnostics plot, Ljung-Box test) to verify
  the model has captured the autocorrelation structure.

In [ ]:
print("Next notebook: ARIMA vs ETS — a head-to-head comparison of the two")
print("major families of time series forecasting methods.")